In [5]:
import os
import pandas as pd
import numpy as np
import statsmodels.api as sm
import glob

## Change input and output paths here

In [ ]:
# Filtering settings
filtering = ["No_Filtering", "More_Than_10_Responses", "More_Than_20_Seconds", "MT10R_AND_MT20S"]
temporal_overlap = ["Start_Date", "Time_of_Day", "Date_and_Time", "No_Time_Control"]
control_for_demo = ["Demographics", "No_Demographics"]
baseline_ERS_use = ["Baseline_ER", "No_Baseline_ER"]

results_folder = "Z:\\Projects\\EMA_Project\\Scripts\\Output\\FO_Results"
os.makedirs(results_folder, exist_ok=True)
input_folder = "Z:\\Projects\\EMA_Project\\Scripts\\Output\\Multiverse_Scratch"

predictors = ['selfER_1', 'selfER_2', 'selfER_3', 'selfER_4', 'selfER_5']
outcomes = ['posAff', 'negAff', 'stress']

In [ ]:
for filter_name in filtering:
    # Load filtered EMA and Intake data
    ema_path = os.path.join(input_folder, f"EMA_data_filtered_{filter_name}.csv")
    intake_path = os.path.join(input_folder, f"Intake_data_temp_{filter_name}.csv")
    if not os.path.exists(ema_path) or not os.path.exists(intake_path):
        print(f"Skipping {filter_name}: files not found.")
        continue

    EMA_data = pd.read_csv(ema_path)
    Intake_data = pd.read_csv(intake_path)

    # Drop missing values for relevant columns
    EMA_data = EMA_data.dropna(subset=predictors + outcomes + ['tempid'])

    # Sex (from Intake_data, merge on tempid)
    EMA_data = EMA_data.merge(Intake_data[['tempid', 'sex']], on='tempid', how='left')

    # Age (from Intake_data)
    EMA_data = EMA_data.merge(Intake_data[['tempid', 'age']], on='tempid', how='left')

    # Demographic variables
    # Fill NAs with 0 first to avoid issues
    for var in ['race_1_black', 'race_2_asian', 'race_3_nathi_pacisl', 'race_5_amin_alnat', 'race_6_other']:
        if var in Intake_data.columns:
            Intake_data[var] = Intake_data[var].fillna(0)

    Intake_data['race_3_other'] = (
        (Intake_data['race_3_nathi_pacisl'] == 1) |
        (Intake_data['race_5_amin_alnat'] == 1) |
        (Intake_data['race_6_other'] == 1)
    ).astype(int)

    # Update demo_vars to use the new combined variable
    demo_vars = [
        'race_1_black', 'race_2_asian', 'eth', 'race_3_other'
    ]

    for var in demo_vars:
        if var in Intake_data.columns:
            EMA_data = EMA_data.merge(Intake_data[['tempid', var]], on='tempid', how='left')

    # EERQ
    erq_mapping = {
        'RP': ['ERQ_RP_01', 'ERQ_RP_02', 'ERQ_RP_03', 'ERQ_RP_04', 'ERQ_RP_05', 'ERQ_RP_06'],
        'DS': ['ERQ_DS_01', 'ERQ_DS_02', 'ERQ_DS_03', 'ERQ_DS_04', 'ERQ_DS_05'],
        'SP': ['ERQ_SP_01', 'ERQ_SP_02', 'ERQ_SP_03', 'ERQ_SP_04'],
        'SA': ['ERQ_SA_01', 'ERQ_SA_02', 'ERQ_SA_03', 'ERQ_SA_04'],
        'SS': ['ERQ_SS_01', 'ERQ_SS_02', 'ERQ_SS_03']
    }

    # List of variables
    EERQ_variables = list(erq_mapping.keys())

    for subscale, columns in erq_mapping.items():
        Intake_data[subscale] = Intake_data[columns].sum(axis=1)
        EMA_data = EMA_data.merge(Intake_data[['tempid', subscale]], on='tempid', how='left')

    # Calculate total ERQ score
    subscales = ['RP', 'SP', 'DS', 'SA', 'SS']
    Intake_data['ERQ_total'] = Intake_data[subscales].sum(axis=1)
    EMA_data = EMA_data.merge(Intake_data[['tempid', 'ERQ_total']], on='tempid', how='left')
    

    # Ensure StartDate is in datetime format and extract the date part
    EMA_data['StartDate'] = pd.to_datetime(EMA_data['StartDate'])
    EMA_data['Day'] = EMA_data['StartDate'].dt.date
    # Compute days since the earliest StartDate in the study
    min_day = EMA_data['Day'].min()
    EMA_data['day0'] = (EMA_data['Day'] - min_day).apply(lambda x: x.days)

    # --- Time of day variable ---
    # Get time as hours since earliest time in the dataset
    EMA_data['TimeOfDay'] = EMA_data['StartDate'].dt.hour + EMA_data['StartDate'].dt.minute / 60.0
    min_time = EMA_data['TimeOfDay'].min()
    EMA_data['TimeOfDay0'] = EMA_data['TimeOfDay'] - min_time

    # --- Multiverse analysis: combinations of covariates and predictors ---
    for demo in control_for_demo:
        for time in temporal_overlap:
            for ER in baseline_ERS_use:
                covariates = []
                RP_covariates = []
                SP_covariates = []
                DS_covariates = []
                SA_covariates = []
                SS_covariates = []

                # Demographics
                if demo == "Demographics":
                    covariates += [v for v in demo_vars if v in EMA_data.columns]
                    RP_covariates += [v for v in demo_vars if v in EMA_data.columns]
                    SP_covariates += [v for v in demo_vars if v in EMA_data.columns]
                    DS_covariates += [v for v in demo_vars if v in EMA_data.columns]
                    SA_covariates += [v for v in demo_vars if v in EMA_data.columns]
                    SS_covariates += [v for v in demo_vars if v in EMA_data.columns]

                # Time
                if time == "Start_Date" or time == "Date_and_Time":
                    covariates += ['day0']
                    RP_covariates += ['day0']
                    SP_covariates += ['day0']
                    DS_covariates += ['day0']
                    SA_covariates += ['day0']
                    SS_covariates += ['day0']

                if time == "Time_of_Day" or time == "Date_and_Time":
                    covariates += ['TimeOfDay0']
                    RP_covariates += ['TimeOfDay0']
                    SP_covariates += ['TimeOfDay0']
                    DS_covariates += ['TimeOfDay0']
                    SA_covariates += ['TimeOfDay0']
                    SS_covariates += ['TimeOfDay0']

                # Baseline ER
                if ER == "Baseline_ER":
                    covariates += ['ERQ_total']
                    RP_covariates += ['ERQ_RP']
                    SP_covariates += ['ERQ_SP']
                    DS_covariates += ['ERQ_DS']
                    SA_covariates += ['ERQ_SA']
                    SS_covariates += ['ERQ_SS']

                # Covariate label for output file naming
                cov_label = f"{demo}_{time}_{ER}"

                # Predictor abbreviation mapping
                predictor_abbr = {
                    'selfER_1': 'RP',
                    'selfER_2': 'SP',
                    'selfER_3': 'DS',
                    'selfER_4': 'SA',
                    'selfER_5': 'SS'
                }                

                # --- Concurrent: all core predictors together ---
                for outcome in outcomes:
                    X = EMA_data[predictors + covariates]
                    y = EMA_data[outcome]
                    tempid = EMA_data['tempid']
                    X = sm.add_constant(X)
                    model = sm.OLS(y, X).fit(cov_type="cluster", cov_kwds={"groups": tempid})
                    summary_df = pd.DataFrame({
                        "Analysis": "Concurrent",
                        "Predictor_Set": "All",
                        "Covariates": cov_label,
                        "Filter": filter_name,
                        "Outcome": outcome,
                        "Predictor": model.params.index,
                        "Coefficient": model.params.values,
                        "CI Lower": model.conf_int()[0].values,
                        "CI Upper": model.conf_int()[1].values,
                        "p-value": model.pvalues.values
                    })
                    summary_df.to_csv(
                        os.path.join(results_folder, f"concurrent_{filter_name}_{outcome}_all_{cov_label}.csv"),
                        index=False
                    )

                # --- Concurrent: single core predictor models ---
                for core in predictors:
                    core_abbr = predictor_abbr.get(core, core)
                    for outcome in outcomes:
                        X = EMA_data[[core] + covariates]
                        y = EMA_data[outcome]
                        tempid = EMA_data['tempid']
                        X = sm.add_constant(X)
                        model = sm.OLS(y, X).fit(cov_type="cluster", cov_kwds={"groups": tempid})
                        summary_df = pd.DataFrame({
                            "Analysis": "Concurrent",
                            "Predictor_Set": core_abbr,  # Use abbreviation
                            "Covariates": cov_label,
                            "Filter": filter_name,
                            "Outcome": outcome,
                            "Predictor": model.params.index,
                            "Coefficient": model.params.values,
                            "CI Lower": model.conf_int()[0].values,
                            "CI Upper": model.conf_int()[1].values,
                            "p-value": model.pvalues.values
                        })
                        summary_df.to_csv(
                            os.path.join(results_folder, f"concurrent_{filter_name}_{outcome}_{core_abbr}_{cov_label}.csv"),  # Use abbreviation
                            index=False
                        )

                # --- Prospective: all core predictors together ---
                X_rows = []
                y_rows = {outcome: [] for outcome in outcomes}
                tempid_rows = []
                for tempid, group in EMA_data.groupby('tempid'):
                    group = group.sort_values('time')
                    if len(group) < 2:
                        continue
                    Xg = group[predictors + covariates].iloc[:-1].reset_index(drop=True)
                    for outcome in outcomes:
                        yg = group[outcome].iloc[1:].reset_index(drop=True)
                        if len(yg) == len(Xg):
                            y_rows[outcome].append(yg)
                    X_rows.append(Xg)
                    tempid_rows.append(pd.Series([tempid] * len(Xg)))
                if X_rows and tempid_rows:
                    X_all = pd.concat(X_rows, ignore_index=True)
                    tempid_all = pd.concat(tempid_rows, ignore_index=True)
                    X_all = sm.add_constant(X_all)
                    for outcome in outcomes:
                        if not y_rows[outcome]:
                            continue
                        y_all = pd.concat(y_rows[outcome], ignore_index=True)
                        if len(np.unique(tempid_all)) < 2 or len(y_all) <= X_all.shape[1]:
                            continue
                        model = sm.OLS(y_all, X_all).fit(cov_type="cluster", cov_kwds={"groups": tempid_all})
                        summary_df = pd.DataFrame({
                            "Analysis": "Prospective",
                            "Predictor_Set": "All",
                            "Covariates": cov_label,
                            "Filter": filter_name,
                            "Outcome": outcome,
                            "Predictor": model.params.index,
                            "Coefficient": model.params.values,
                            "CI Lower": model.conf_int()[0].values,
                            "CI Upper": model.conf_int()[1].values,
                            "p-value": model.pvalues.values,
                        })
                        summary_df.to_csv(
                            os.path.join(results_folder, f"prospective_{filter_name}_{outcome}_all_{cov_label}.csv"),
                            index=False
                        )

                # --- Prospective: single core predictor models ---
                for core in predictors:
                    core_abbr = predictor_abbr.get(core, core)
                    X_rows = []
                    y_rows = {outcome: [] for outcome in outcomes}
                    tempid_rows = []
                    for tempid, group in EMA_data.groupby('tempid'):
                        group = group.sort_values('time')
                        if len(group) < 2:
                            continue
                        Xg = group[[core] + covariates].iloc[:-1].reset_index(drop=True)
                        for outcome in outcomes:
                            yg = group[outcome].iloc[1:].reset_index(drop=True)
                            if len(yg) == len(Xg):
                                y_rows[outcome].append(yg)
                        X_rows.append(Xg)
                        tempid_rows.append(pd.Series([tempid] * len(Xg)))
                    if X_rows and tempid_rows:
                        X_all = pd.concat(X_rows, ignore_index=True)
                        tempid_all = pd.concat(tempid_rows, ignore_index=True)
                        X_all = sm.add_constant(X_all)
                        for outcome in outcomes:
                            if not y_rows[outcome]:
                                continue
                            y_all = pd.concat(y_rows[outcome], ignore_index=True)
                            if len(np.unique(tempid_all)) < 2 or len(y_all) <= X_all.shape[1]:
                                continue
                            y_all = pd.concat(y_rows[outcome], ignore_index=True)
                            if len(np.unique(tempid_all)) < 2 or len(y_all) <= X_all.shape[1]:
                                continue
                            model = sm.OLS(y_all, X_all).fit(cov_type="cluster", cov_kwds={"groups": tempid_all})
                            summary_df = pd.DataFrame({
                                "Analysis": "Prospective",
                                "Predictor_Set": core_abbr,  # Use abbreviation
                                "Covariates": cov_label,
                                "Filter": filter_name,
                                "Outcome": outcome,
                                "Predictor": model.params.index,
                                "Coefficient": model.params.values,
                                "CI Lower": model.conf_int()[0].values,
                                "CI Upper": model.conf_int()[1].values,
                                "p-value": model.pvalues.values,
                            })
                            summary_df.to_csv(
                                os.path.join(results_folder, f"prospective_{filter_name}_{outcome}_{core_abbr}_{cov_label}.csv"),  # Use abbreviation
                                index=False
                            )
print("All multiverse regressions complete.")

In [8]:
# Set up FO_graph output folder
fo_graph_folder = os.path.join(results_folder, "..", "FO_graph")
os.makedirs(fo_graph_folder, exist_ok=True)

# Mapping: desired label -> actual label in FO_results
strategies_map = {'RP': "selfER_1",
                  'SP': "selfER_2",
                  'DS': "selfER_3",
                  'SA': "selfER_4",
                  'SS': "selfER_5"}
outcomes = ['posAff', 'negAff', 'stress']

# 1. Bind all single-predictor results for each outcome-strategy pair
for outcome in outcomes:
    for label, actual in strategies_map.items():
        pattern = os.path.join(results_folder, f"*_{outcome}_{label}_*.csv")
        files = glob.glob(pattern)
        dfs = []
        for f in files:
            df = pd.read_csv(f)
            # Only keep rows for the actual label in FO_results
            df_strat = df[df["Predictor"] == actual].copy()
            df_strat["Result_File"] = os.path.basename(f)
            if not df_strat.empty:
                dfs.append(df_strat)
        if dfs:
            all_single = pd.concat(dfs, ignore_index=True)
            all_single.to_csv(os.path.join(fo_graph_folder, f"FO_{outcome}_{label}_single.csv"), index=False)

# 2. Bind all joint regression results for each outcome-strategy pair
joint_pattern = os.path.join(results_folder, f"*_all_*.csv")
joint_files = glob.glob(joint_pattern)
joint_results = {(outcome, label): [] for outcome in outcomes for label in strategies_map}

for f in joint_files:
    df = pd.read_csv(f)
    for outcome in outcomes:
        for label, actual in strategies_map.items():
            # Only keep rows for the actual label in FO_results
            df_strat = df[(df["Outcome"] == outcome) & (df["Predictor"] == actual)].copy()
            df_strat["Result_File"] = os.path.basename(f)
            if not df_strat.empty:
                joint_results[(outcome, label)].append(df_strat)

for outcome in outcomes:
    for label in strategies_map:
        if joint_results[(outcome, label)]:
            all_joint = pd.concat(joint_results[(outcome, label)], ignore_index=True)
            all_joint.to_csv(os.path.join(fo_graph_folder, f"FO_{outcome}_{label}_joint.csv"), index=False)

print("FO summary files for each outcome-strategy pair and strategy saved in FO_graph folder.")

FO summary files for each outcome-strategy pair and strategy saved in FO_graph folder.
